In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import pytz
import yfinance as yf
import pyodbc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time
import logging
import pandas as pd
from truedata import TD_hist
import requests

In [2]:
def fetch_truedata_history(
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData for a list of tickers.

    Parameters
    ----------
    username : str
        TrueData username.
    password : str
        TrueData password.
    ticker_list : list
        List of ticker symbols to fetch data for.
    duration : str, optional
        Duration of data (e.g., '1 Y', '25 Y', etc.). Default is '1 Y'.
    bar_size : str, optional
        Bar size for data ('EOD', 'WEEK', etc.). Default is 'EOD'.
    sleep_time : float, optional
        Delay between API calls to avoid throttling. Default is 0.2 seconds.

    Returns
    -------
    final_df : pd.DataFrame
        Combined DataFrame of all tickers' historical data.
    error_list : list
        List of tickers that failed to fetch.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    username = 'tdwsf695'
    password = 'ocean@695'
    # Initialize connection
    td_hist = TD_hist(username, password)
    df_list = []
    error_list = []
    for ticker in ticker_list:
        try:
            df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

            df['Ticker'] = ticker
            # Check column names and rename accordingly
            rename_dict = {}
            if 'timestamp' in df.columns:
                rename_dict['timestamp'] = 'Date'
            elif 'datetime' in df.columns:
                rename_dict['datetime'] = 'Date'
            elif 'date' in df.columns:
                rename_dict['date'] = 'Date'
            rename_dict.update({
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'open': 'Open'
            })
            df = df.rename(columns=rename_dict)

            df_list.append(df)
            logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
            time.sleep(sleep_time)

        except Exception as e:
            logging.error(f"Failed to fetch data for {ticker}: {e}")
            error_list.append(ticker)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list

In [3]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

def process_portfolio(nav_df, ticker_data, initial_value=75, output_file=None):
    """
    Process portfolio allocation and returns final dataframe with portfolio performance.

    Parameters
    ----------
    nav_df : pd.DataFrame
        Dataframe with at least ['Year-Month', 'Ticker'] columns.
    get_individual_stock_data : function
        Function to fetch OHLC data. Must accept (tickers, start_date, end_date) and return DataFrame with ['Date','Ticker','Close'].
    initial_value : float
        Initial portfolio allocation value (default=75).
    debt_ticker : str
        Ticker used as debt/alternative asset (default 'MOGSEC.NS').
    output_file : str or None
        If provided, saves the final dataframe to Excel.

    Returns
    -------
    pd.DataFrame
        Final dataframe with portfolio values.
    """
    df_lis = []
    last_month_value = {}
    year_months = nav_df['Year-Month'].unique()
    for i, year_month in enumerate(year_months):
        print(f"\nProcessing: {year_month}")
        print("Last Month Value:", last_month_value)

        tickers = nav_df[nav_df['Year-Month'] == year_month]['Ticker'].unique()
        year_month_date = pd.to_datetime(f"{year_month}-01")


        prev_month_start = (year_month_date - relativedelta(months=2)).strftime('%Y-%m-%d')
        curr_month_start = year_month_date.strftime('%Y-%m-%d')
        curr_month_end = (year_month_date + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

        # --- Fetch stock data ---
        # stock_data = get_individual_stock_data(tickers, prev_month_start, curr_month_end)
        stock_data = (
            ticker_data[(ticker_data['Date'] >= prev_month_start)
            & (ticker_data['Date'] <= curr_month_end) 
            & (ticker_data['Ticker'].isin(tickers))])
        
        # % change
        stock_data['%change'] = stock_data.groupby('Ticker')['Close'].pct_change()
        # Filter current month
        stock_data_flt = stock_data[
            (stock_data['Date'] >= curr_month_start) & (stock_data['Date'] <= curr_month_end)
        ].copy()

        print(stock_data_flt)

        # --- Portfolio allocation logic ---
        if len(last_month_value) == 0:
            # First month → allocate initial portfolio equally
            allocation_per_stock = initial_value / len(tickers)
            stock_allocations = {t: allocation_per_stock for t in tickers}
        else:
            # Continue portfolio
            stock_allocations = {t: last_month_value[t] for t in tickers if t in last_month_value}

            # Pool value of dropped stocks
            dropped_stocks = [t for t in last_month_value if t not in tickers]
            dropped_value = sum(last_month_value[t] for t in dropped_stocks)

            # New stocks → share the dropped value equally
            new_stocks = [t for t in tickers if t not in last_month_value]
            if new_stocks:
                allocation_per_stock = dropped_value / len(new_stocks)
                for t in new_stocks:
                    stock_allocations[t] = allocation_per_stock

        # Apply allocations into dataframe
        for tkr, init_value in stock_allocations.items():
            tkr_idx = stock_data_flt[stock_data_flt['Ticker'] == tkr].index
            stock_data_flt.loc[tkr_idx, 'Buy_Hold_Value'] = init_value * (
                (1 + stock_data_flt.loc[tkr_idx, '%change'].fillna(0)).cumprod()
            )

        # --- Update last month values ---
        last_month_value = (
            stock_data_flt.groupby('Ticker')['Buy_Hold_Value'].last().to_dict()
        )

        # --- Track total portfolio value ---
        stock_data_flt['Total_Portfolio_Value'] = (
            stock_data_flt.groupby('Date')['Buy_Hold_Value'].transform('sum')
        )

        df_lis.append(stock_data_flt)

    
    final_df = pd.concat(df_lis).reset_index(drop=True)

    if output_file:
        final_df.to_excel(output_file, index=False)

    return final_df


In [4]:
import os
import pandas as pd

def prepare_and_process_portfolio(input_file, start_date, end_date, output_folder,
                                  process_portfolio,
                                  equity_allocation=75, gold_allocation=25):
    """
    Prepare portfolio dataframe with momentum stocks + GOLDBEES and process performance.

    Parameters
    ----------
    input_file : str
        Path to momentum Excel file (with End_Date, Ticker columns).
    start_date : str (YYYY-MM-DD)
        Start date for filtering.
    end_date : str (YYYY-MM-DD)
        End date for filtering.
    output_folder : str
        Folder to save output file.
    get_individual_stock_data : function
        Function to fetch stock NAV/price data.
    process_pocrtfolio : function
        Function to process equity portion of portfolio.
    process_gold : function
        Function to process gold portion of portfolio.
    equity_allocation : int, optional
        Initial allocation to equities (default=75000).
    gold_allocation : int, optional
        Initial allocation to gold (default=25000).

    Returns
    -------
    final_df : pd.DataFrame
        Combined portfolio dataframe.
    """

    # Load and clean
    nav_df = pd.read_excel(input_file).rename(columns={'End_Date': 'Date'})
    nav_df['Date'] = pd.to_datetime(nav_df['Date'])
    nav_df = (
        nav_df[(nav_df['Date'] >= start_date) & (nav_df['Date'] <= end_date)]
        .reset_index(drop=True)[['Date', 'Ticker']]
    )
    nav_df['Year-Month'] = nav_df['Date'].dt.to_period('M').astype(str)
    stocks = pd.read_excel(input_file)

    # Add GOLDBEES for each unique date
    goldbees_df = pd.DataFrame({
        'Date': nav_df['Date'].unique(),
        'Ticker': 'GOLDBEES'
    })
    goldbees_df['Year-Month'] = pd.to_datetime(goldbees_df['Date']).dt.to_period('M').astype(str)
    # print(goldbees_df)

    # Combine
    concat_df = (
        pd.concat([nav_df, goldbees_df], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )

    # symbol_list = stocks['Ticker'].unique()
    # ticker_data = fetch_truedata_history(
    #     ticker_list = symbol_list,
    #     duration = '5 Y',
    #     bar_size = 'EOD',
    #     sleep_time= 0.1
    # )[0]
    # final_df = process_portfolio(concat_df, ticker_data, equity_allocation)

    
    # Split
    ticker_df = concat_df.query("Ticker != 'GOLDBEES'")
    symbol_list = ticker_df['Ticker'].unique()
    ticker_data_other_stocks = fetch_truedata_history(
        ticker_list = symbol_list,
        duration = '10 Y',
        bar_size = 'EOD',
        sleep_time= 0.1
    )[0]

    
    gold_df = concat_df.query("Ticker == 'GOLDBEES'")
    symbol_list = gold_df['Ticker'].unique()
    ticker_data_gold = fetch_truedata_history(
        ticker_list = symbol_list,
        duration = '10 Y',
        bar_size = 'EOD',
        sleep_time= 0.1
    )[0]
    # print(gold_df)

    # Process
    final_df_other_stocks = process_portfolio(ticker_df, ticker_data_other_stocks, equity_allocation)
    # final_df_gold = process_gold(gold_df, get_individual_stock_data, gold_allocation)
    final_df_gold = process_portfolio(gold_df, ticker_data_gold, gold_allocation)


    # Merge results
    final_df = (
        pd.concat([final_df_other_stocks, final_df_gold], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )


    # --- ensure output folder exists ---
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # --- extract middle folder name from input path ---
    middle_folder = os.path.basename(os.path.dirname(input_file))
    # e.g. for path ".../nifty500_21April2025_results/master_momentum_summary.xlsx"
    # middle_folder = "nifty500_21April2025_results"

    # --- create output filename using middle folder ---
    output_file = os.path.join(output_folder, f"{middle_folder}_gold_buy&hold_returns.xlsx")

    # --- save output ---
    # final_df.to_excel(output_file, index=False)
    print(f"✅ Final output saved to: {output_file}")

    return final_df


In [5]:
#NSE500

In [6]:
final_df = prepare_and_process_portfolio(
    input_file="Stocks/Nifty_500_2025_Apr_20_stocks_results/master_momentum_summary.xlsx",
    start_date="2023-04-01",
    end_date=date.today().strftime('%Y-%m-%d'),
    output_folder="Trials",
    process_portfolio=process_portfolio
)

import plotly.express as px

# ✅ Group by Date and calculate total portfolio value
portfolio_summary = (
    final_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)

# ✅ Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # 🔑 width
                  height=500)    # 🔑 height

fig.show()

(2026-01-22 09:47:37,269) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
2026-01-22 09:47:37,269 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-01-22 09:47:37,620 - INFO - Fetched data for ABCAPITAL (2081 rows).
2026-01-22 09:47:38,001 - INFO - Fetched data for ANANDRATHI (1021 rows).
2026-01-22 09:47:38,431 - INFO - Fetched data for BANKBARODA (2477 rows).
2026-01-22 09:47:38,855 - INFO - Fetched data for BANKINDIA (2477 rows).
2026-01-22 09:47:39,288 - INFO - Fetched data for BOSCHLTD (2477 rows).
2026-01-22 09:47:39,720 - INFO - Fetched data for CANBK (2477 rows).
2026-01-22 09:47:40,167 - INFO - Fetched data for CUMMINSIND (2477 rows).
2026-01-22 09:47:40,640 - INFO - Fetched data for FINCABLES (2477 rows).
2026-01-22 09:47:41,095 - INFO - Fetched data for GODFRYPHLP (2477 rows).
2026-01-22 09:47:41,535 - INFO - Fetched data for J&KBANK (2477 rows).
2026-01-22 09:47:42,025 - INFO - Fetched data for


Processing: 2023-04
Last Month Value: {}
            Date    Open    High     Low   Close   volume  oi     Ticker  \
1384  2023-04-03  154.00  154.90  152.00  153.80  2347046   0  ABCAPITAL   
1385  2023-04-05  153.80  156.45  152.80  155.25  3010848   0  ABCAPITAL   
1386  2023-04-06  155.70  159.30  153.80  157.90  3718974   0  ABCAPITAL   
1387  2023-04-10  158.75  160.30  155.90  159.75  3508046   0  ABCAPITAL   
1388  2023-04-11  160.50  161.30  157.05  157.60  2936298   0  ABCAPITAL   
...          ...     ...     ...     ...     ...      ...  ..        ...   
45314 2023-04-24  332.50  336.80  324.45  331.60   867129   0   TITAGARH   
45315 2023-04-25  330.00  352.90  329.05  336.85  4114109   0   TITAGARH   
45316 2023-04-26  339.00  342.70  329.15  332.35   709013   0   TITAGARH   
45317 2023-04-27  332.85  339.55  323.95  331.50   794444   0   TITAGARH   
45318 2023-04-28  333.00  335.20  329.25  332.25   351839   0   TITAGARH   

        %change  
1384   0.001628  
1385   0.

In [7]:
final_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564,0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923,0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
14632,2026-01-22,242.60,248.20,242.50,246.90,400919,0,NYKAA,0.021726,9.589283,216.321481
14633,2026-01-22,2849.20,2881.50,2700.00,2869.30,68639,0,RADICO,0.013028,9.138293,216.321481
14634,2026-01-22,1035.00,1055.50,1034.05,1049.00,3197476,0,SBIN,0.019783,11.219358,216.321481
14635,2026-01-22,995.05,1005.40,986.10,1002.90,759014,0,SHRIRAMFIN,0.017140,10.575565,216.321481


In [8]:
old_df = final_df[~((final_df['Date']>'2025-11-30') & (final_df['Ticker']=='GOLDBEES'))]
old_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564,0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923,0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
14632,2026-01-22,242.60,248.20,242.50,246.90,400919,0,NYKAA,0.021726,9.589283,216.321481
14633,2026-01-22,2849.20,2881.50,2700.00,2869.30,68639,0,RADICO,0.013028,9.138293,216.321481
14634,2026-01-22,1035.00,1055.50,1034.05,1049.00,3197476,0,SBIN,0.019783,11.219358,216.321481
14635,2026-01-22,995.05,1005.40,986.10,1002.90,759014,0,SHRIRAMFIN,0.017140,10.575565,216.321481


In [9]:
np.sort(old_df['Ticker'].unique())

array(['AAVAS', 'ABB', 'ABCAPITAL', 'ADANIPORTS', 'AIIL', 'AJANTPHARM',
       'AMBER', 'ANANDRATHI', 'ANANTRAJ', 'ANGELONE', 'APARINDS',
       'APLAPOLLO', 'ASTERDM', 'AUBANK', 'AUROPHARMA', 'BAJAJ-AUTO',
       'BAJAJHLDNG', 'BAJFINANCE', 'BANKBARODA', 'BANKINDIA', 'BASF',
       'BDL', 'BEL', 'BEML', 'BERGEPAINT', 'BHARTIARTL', 'BHARTIHEXA',
       'BHEL', 'BLUESTARCO', 'BOSCHLTD', 'BPCL', 'BSE', 'BSOFT', 'CANBK',
       'CHAMBLFERT', 'CHENNPETRO', 'CHOLAFIN', 'CHOLAHLDNG', 'COCHINSHIP',
       'COFORGE', 'COHANCE', 'COLPAL', 'CONCORDBIO', 'COROMANDEL', 'CUB',
       'CUMMINSIND', 'CYIENT', 'DATAPATTNS', 'DBREALTY', 'DEEPAKFERT',
       'DELHIVERY', 'DIVISLAB', 'DIXON', 'DLF', 'DOMS', 'ECLERX',
       'EICHERMOT', 'EIHOTEL', 'ELECON', 'EMAMILTD', 'ENGINERSIN', 'ERIS',
       'ETERNAL', 'EXIDEIND', 'FINCABLES', 'FLUOROCHEM', 'FORTIS',
       'GLAND', 'GLENMARK', 'GMDCLTD', 'GODFRYPHLP', 'GODREJPROP',
       'GOLDBEES', 'GPIL', 'GRAVITA', 'GVT&D', 'HAL', 'HBLENGINE',
       'HDFCBANK

In [10]:
df = fetch_truedata_history(
    ticker_list = ['GOLDBEES', 'SILVERBEES', 'MOGSEC'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
df = df[["Date","Ticker", "Open", "Close"]]
df['%change'] = df['Close'].pct_change()
df = df[df['Date'] >= '2025-12-01']
# df.to_excel('C:\\Users\\Admin\\Momentum\\Automating Momentum True Data\\Trials\\nse200_Nifty_200_2025_Aug_nse200_nse200_nse200_nse200_nse200_returns.xlsx', index=False)
df

(2026-01-22 09:49:24,497) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
(2026-01-22 09:49:24,497) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
(2026-01-22 09:49:24,497) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
2026-01-22 09:49:24,497 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-01-22 09:49:24,814 - INFO - Fetched data for GOLDBEES (1240 rows).
2026-01-22 09:49:25,232 - INFO - Fetched data for SILVERBEES (983 rows).
2026-01-22 09:49:25,640 - INFO - Fetched data for MOGSEC (1144 rows).


,Date,Ticker,Open,Close,%change
1203,2025-12-01,GOLDBEES,106.09,106.72,0.020073
1204,2025-12-02,GOLDBEES,106.65,105.63,-0.010214
1205,2025-12-03,GOLDBEES,106.25,106.51,0.008331
1206,2025-12-04,GOLDBEES,106.67,105.92,-0.005539
1207,2025-12-05,GOLDBEES,106.27,106.89,0.009158
...,...,...,...,...,...
3362,2026-01-14,MOGSEC,62.89,62.99,0.000159
3363,2026-01-16,MOGSEC,62.90,62.70,-0.004604
3364,2026-01-19,MOGSEC,62.39,62.72,0.000319
3365,2026-01-20,MOGSEC,62.61,62.60,-0.001913


In [11]:
ticker_weights = {'GOLDBEES':0.6, 'SILVERBEES':0.2, 'MOGSEC':0.2}
factor = 51.2140199725867

ticker_value = {ticker: weight * factor for ticker, weight in ticker_weights.items()}
ticker_value

{'GOLDBEES': 30.728411983552018,
 'SILVERBEES': 10.24280399451734,
 'MOGSEC': 10.24280399451734}

In [12]:
df['BaseValue'] = df['Ticker'].map(ticker_value)
# Assign values
df['Value'] = df['Ticker'].map(ticker_value)
# Convert %change to numeric (if needed)
df['%change'] = pd.to_numeric(df['%change'])
# Sort (important for cumprod)
df = df.sort_values(['Ticker', 'Date'])

# Daily growth factor
df['ret_factor'] = 1 + df['%change']

# Cumulative factor per ticker
df['cum_factor'] = df.groupby('Ticker')['ret_factor'].cumprod()

# FINAL DAILY VALUE
df['Value_On_Date'] = df['BaseValue'] * df['cum_factor']

df = df[['Date', 'Ticker', 'Open', 'Close', 'Value_On_Date', '%change']].rename(columns={'Value_On_Date':'Buy_Hold_Value'})
df

,Date,Ticker,Open,Close,Buy_Hold_Value,%change
1203,2025-12-01,GOLDBEES,106.09,106.72,31.345212,0.020073
1204,2025-12-02,GOLDBEES,106.65,105.63,31.025064,-0.010214
1205,2025-12-03,GOLDBEES,106.25,106.51,31.283532,0.008331
1206,2025-12-04,GOLDBEES,106.67,105.92,31.110241,-0.005539
1207,2025-12-05,GOLDBEES,106.27,106.89,31.395144,0.009158
...,...,...,...,...,...,...
2218,2026-01-16,SILVERBEES,262.81,269.44,17.604268,0.025227
2219,2026-01-19,SILVERBEES,274.30,282.95,18.486964,0.050141
2220,2026-01-20,SILVERBEES,288.15,297.62,19.445451,0.051847
2221,2026-01-21,SILVERBEES,300.00,310.68,20.298746,0.043881


In [13]:
conc_df = pd.concat([old_df, df])
conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046.0,0.0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564.0,0.0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059.0,0.0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076.0,0.0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923.0,0.0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
2218,2026-01-16,262.81,NaN,NaN,269.44,NaN,NaN,SILVERBEES,0.025227,17.604268,NaN
2219,2026-01-19,274.30,NaN,NaN,282.95,NaN,NaN,SILVERBEES,0.050141,18.486964,NaN
2220,2026-01-20,288.15,NaN,NaN,297.62,NaN,NaN,SILVERBEES,0.051847,19.445451,NaN
2221,2026-01-21,300.00,NaN,NaN,310.68,NaN,NaN,SILVERBEES,0.043881,20.298746,NaN


In [14]:
import plotly.express as px

# ✅ Group by Date and calculate total portfolio value
portfolio_summary = (
    conc_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)
# ✅ Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # 🔑 width
                  height=500)    # 🔑 height
fig.show()

In [15]:
# Momentum/Automating Momentum True Data/Trials/Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx

In [16]:
conc_df.to_excel('C:\\Users\\anike\\Desktop\\Ocean_dev\\Momentum Handover\\Momentum Handover\\Trials\\Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx', index=False)
# \Trials

In [17]:
nse = fetch_truedata_history(
    ticker_list = ['Nifty 500'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
nse = nse[["Date", "Close"]].rename(columns={'Close':'Buy_Hold_Value'})
nse['%change'] = nse['Buy_Hold_Value'].pct_change()
nse = nse[nse['Date'] >= '2023-04-01']
nse.to_excel('Trials\\nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_returns.xlsx', index=False)
nse

(2026-01-22 09:49:27,975) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
(2026-01-22 09:49:27,975) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
(2026-01-22 09:49:27,975) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
(2026-01-22 09:49:27,975) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:14848 Thread:20300)
2026-01-22 09:49:27,975 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-01-22 09:49:28,295 - INFO - Fetched data for Nifty 500 (1241 rows).


,Date,Buy_Hold_Value,%change
544,2023-04-03,14601.95,0.003029
545,2023-04-05,14709.40,0.007359
546,2023-04-06,14759.20,0.003386
547,2023-04-10,14790.55,0.002124
548,2023-04-11,14867.25,0.005186
...,...,...,...
1236,2026-01-16,23485.30,0.000403
1237,2026-01-19,23373.75,-0.004750
1238,2026-01-20,22946.65,-0.018273
1239,2026-01-21,22835.15,-0.004859


In [18]:
conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046.0,0.0,ABCAPITAL,0.001628,3.756106,75.779854
1,2023-04-03,407.00,410.95,403.05,405.15,112564.0,0.0,ANANDRATHI,0.003219,3.762071,75.779854
2,2023-04-03,169.10,170.25,168.10,169.20,16659059.0,0.0,BANKBARODA,0.002073,3.757773,75.779854
3,2023-04-03,75.00,76.45,74.10,75.95,8102076.0,0.0,BANKINDIA,0.017415,3.815305,75.779854
4,2023-04-03,19500.00,19500.85,19200.00,19434.25,29923.0,0.0,BOSCHLTD,0.003322,3.762458,75.779854
...,...,...,...,...,...,...,...,...,...,...,...
2218,2026-01-16,262.81,NaN,NaN,269.44,NaN,NaN,SILVERBEES,0.025227,17.604268,NaN
2219,2026-01-19,274.30,NaN,NaN,282.95,NaN,NaN,SILVERBEES,0.050141,18.486964,NaN
2220,2026-01-20,288.15,NaN,NaN,297.62,NaN,NaN,SILVERBEES,0.051847,19.445451,NaN
2221,2026-01-21,300.00,NaN,NaN,310.68,NaN,NaN,SILVERBEES,0.043881,20.298746,NaN
